# Explainable AI: COMPAS Predictions

Importing libraries

In [ ]:
import pandas as pd
import numpy as np

# for imputing modes and medians
from sklearn.impute import SimpleImputer

# for random forest
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# for dummy model (baseline)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report


# ! Assign target !

In [ ]:
target = 'is_recid'

Importing data

In [ ]:
# change file name for data if using different version
dfOriginal = pd.read_csv("cox-violent-parsed_filt.csv")

Remove duplicates, only one row per name

In [ ]:
dfProcessed = dfOriginal.drop_duplicates(subset=['name'])
dfProcessed.count()

,0
id,6560
name,10855
first,10855
last,10855
sex,10855
dob,10855
age,10855
age_cat,10855
race,10855
juv_fel_count,10855


Remove unused columns

In [ ]:
dfProcessed = dfProcessed[['sex', 'age', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree', target]]
dfProcessed.count()

,0
sex,10855
age,10855
race,10855
juv_fel_count,10855
juv_misd_count,10855
juv_other_count,10855
priors_count,10855
c_charge_degree,10185
is_recid,10855


Remove -1 is_recid (must be binary)

In [ ]:
# What we expect to be dropped
# Just using this to double check
dfCheck = dfProcessed.loc[dfProcessed[target] < 0]
dfCheck.count()

,0
sex,648
age,648
race,648
juv_fel_count,648
juv_misd_count,648
juv_other_count,648
priors_count,648
c_charge_degree,1
is_recid,648


In [ ]:
# Dropping the invalid is_recid values
dfProcessed = dfProcessed.loc[dfProcessed[target] > -1]
dfProcessed.count()

,0
sex,10207
age,10207
race,10207
juv_fel_count,10207
juv_misd_count,10207
juv_other_count,10207
priors_count,10207
c_charge_degree,10184
is_recid,10207


Missing value strategy
1. Numerical values --> MEDIAN imputation
2. Categorical values --> MODE imputation

In [ ]:
print("Any NaN values?\n", dfProcessed.isnull().values.any())
# ^^ To check if any NaNs
# put it bc I tried a heatmap and it looked empty so double checking

Any NaN values?
 True


# DATA LEAKAGE 😰😰😰😰

In [ ]:
# Numerical columns
numCols = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']
numImputer = SimpleImputer(strategy = 'median')
dfProcessed[numCols] = numImputer.fit_transform(dfProcessed[numCols])

# Categorical columns
catCols = ['sex', 'race', 'c_charge_degree']
catImputer = SimpleImputer(strategy = 'most_frequent') # most_frequent = mode
dfProcessed[catCols] = catImputer.fit_transform(dfProcessed[catCols])

print("Any NaN values?\n", dfProcessed.isnull().values.any())
# no more NaNs :)

Any NaN values?
 False


Smote? Impute from other dataset? Impute from same dataset?

In [ ]:
# We should discuss what the plan is for this part

## Random Forest
!Note!

Random Forest requires all data to be in numbers, so the categorical data needs to be enumerated before we are able to use it. This is fine for our situation because the categorical data is not unique to each individual and can easily be turned into numbers.

In [ ]:
# get_dummies is a tool for encoding categorical columns
X = pd.get_dummies(dfProcessed.drop(columns=[target]))
y = dfProcessed[target]

testSize = 0.3 # change this variable if you want different train/test split
randNum = 44 # change if you want to adjust the random_state

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=testSize, random_state=randNum)

rf = RandomForestClassifier(n_estimators=300, random_state=randNum, min_samples_leaf=5, max_depth=10, class_weight='balanced')
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=5, n_estimators=300, random_state=44)

In [ ]:
# What did the model predict?
predictions = rf.predict(X_test)
predictions

array([0, 1, 0, ..., 1, 1, 0])

In [ ]:
# The probabilities it gave to each prediction
rf.predict_proba(X_test)

array([[0.5792188 , 0.4207812 ],
       [0.34951405, 0.65048595],
       [0.71898156, 0.28101844],
       ...,
       [0.30348349, 0.69651651],
       [0.49229964, 0.50770036],
       [0.77061865, 0.22938135]])

In [ ]:
# Importance of each feature
importances = rf.feature_importances_
cols = X.columns

for i in range(len(cols)):
    print(f'Importance of {cols[i]} = {round(importances[i] * 100, 2)}%.')

# Note: the categorical features are split into
#   separate cols from using the dummy tool earlier.
# I kept them separate for now so we can see how
#   different races and charge degrees are treated.

Importance of age = 28.45%.
Importance of juv_fel_count = 2.66%.
Importance of juv_misd_count = 4.8%.
Importance of juv_other_count = 5.53%.
Importance of priors_count = 38.59%.
Importance of sex_Female = 2.13%.
Importance of sex_Male = 2.48%.
Importance of race_African-American = 4.26%.
Importance of race_Asian = 0.26%.
Importance of race_Caucasian = 1.63%.
Importance of race_Hispanic = 1.09%.
Importance of race_Native American = 0.03%.
Importance of race_Other = 0.68%.
Importance of c_charge_degree_(CO3) = 0.0%.
Importance of c_charge_degree_(CT) = 0.0%.
Importance of c_charge_degree_(F1) = 0.88%.
Importance of c_charge_degree_(F2) = 1.09%.
Importance of c_charge_degree_(F3) = 2.32%.
Importance of c_charge_degree_(F5) = 0.0%.
Importance of c_charge_degree_(F6) = 0.01%.
Importance of c_charge_degree_(F7) = 0.5%.
Importance of c_charge_degree_(M1) = 1.48%.
Importance of c_charge_degree_(M2) = 0.93%.
Importance of c_charge_degree_(MO3) = 0.17%.
Importance of c_charge_degree_(NI0) = 0.0%

In [ ]:
# Classification report returns precision, recall, F1, etc.
from sklearn.metrics import classification_report
print(classification_report(y_test, rf.predict(X_test)))

              precision    recall  f1-score   support

           0       0.79      0.66      0.72      2047
           1       0.49      0.65      0.56      1016

    accuracy                           0.66      3063
   macro avg       0.64      0.66      0.64      3063
weighted avg       0.69      0.66      0.67      3063



# Dummy model (always is_recid = 0)

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_dummy_pred = dummy.predict(X_test)

print(classification_report(y_test, y_dummy_pred))

              precision    recall  f1-score   support

           0       0.67      1.00      0.80      2047
           1       0.00      0.00      0.00      1016

    accuracy                           0.67      3063
   macro avg       0.33      0.50      0.40      3063
weighted avg       0.45      0.67      0.54      3063



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Comparison of the two models (by FPR)

In [ ]:
df_eval = pd.DataFrame({
    "y_true": y_test,
    "race": dfProcessed.loc[y_test.index, "race"]
})

df_eval["rf_pred"] = rf.predict(X_test)
df_eval["dummy_pred"] = dummy.predict(X_test)

def false_positive_rate(y_true, y_pred):
    fp = np.sum((y_pred == 1) & (y_true == 0))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    return fp / (fp + tn)

fpr_rf = false_positive_rate(df_eval["y_true"], df_eval["rf_pred"])
fpr_dummy = false_positive_rate(df_eval["y_true"], df_eval["dummy_pred"])

print(f"Random Forest FPR: {fpr_rf:.3f}")
print(f"Dummy Model FPR:   {fpr_dummy:.3f}")

Random Forest FPR: 0.340
Dummy Model FPR:   0.000


In [ ]:
fpr_by_race = {}

for race in df_eval["race"].unique():
    subset = df_eval[df_eval["race"] == race]
    fpr_by_race[race] = {
        "RandomForest": false_positive_rate(
            subset["y_true"], subset["rf_pred"]
        ),
        "Dummy": false_positive_rate(
            subset["y_true"], subset["dummy_pred"]
        )
    }

pd.DataFrame(fpr_by_race).T

,RandomForest,Dummy
Hispanic,0.183036,0.0
Other,0.188976,0.0
Caucasian,0.253846,0.0
African-American,0.481069,0.0
Asian,0.000000,0.0
Native American,0.222222,0.0


# TreeSHAP

In [ ]:
import shap

explainer = shap.TreeExplainer(rf)

shapValues = explainer.shap_values(X_test)

# ANCHORS

In [ ]:
# anchors